# 第 3 章 · 工具系统：让 Agent 从"会说"到"会做"

> LLM 本身只有"语言智能"，而真实任务需要"行动力"——查数据、算数学、调 API。**工具（Tool）就是 Agent 的手和脚**。本章覆盖：
> 1. `FunctionTool`：把 Python 函数变成 Agent 的技能；
> 2. `ToolContext`：工具与 Session 状态的双向通道；
> 3. `AgentTool`：把另一个 Agent 封装成工具；
> 4. 内置工具、MCP 与第三方工具生态。

---

## 1. 工具调用在事件流中的样子

第 1 章说过：理解事件流就理解了 ADK 的一半。工具调用会让事件流多出两个关键环节：

```mermaid
sequenceDiagram
    participant U as 用户
    participant A as Agent(LLM)
    participant F as 你的Python函数
    U->>A: 北京明天天气怎么样？
    Note over A: LLM 判断：需要调用工具
    A-->>U: Event① function_call:<br/>get_weather(city="北京")
    A->>F: 执行函数
    F-->>A: 返回 {"天气": "晴"}
    A-->>U: Event② function_response:<br/>工具结果
    Note over A: LLM 基于结果组织语言
    A-->>U: Event③ 最终文本回复
```

> 📌 **重要认知**：函数是**框架替你执行的**，LLM 只负责"决定调什么、传什么参"和"解读结果"。函数里的代码运行在**你的进程**里，可以访问数据库、内网 API 等一切资源。

---

## 2. FunctionTool：函数即工具

ADK 的做法极简——**把普通 Python 函数放进 `tools` 列表**即可（框架自动包装为 `FunctionTool`）。约定：

| 要素 | 来源 | 作用 |
|---|---|---|
| 工具名 | 函数名 | LLM 选择工具时看到的名字 |
| 工具描述 | **docstring** | LLM 判断"何时用"的依据，**必须写清楚** |
| 参数 schema | **类型注解** | LLM 生成调用参数的依据 |
| 返回值 | dict 最佳 | 结构化结果便于 LLM 解读 |

下面给 Agent 配两个工具，并把事件流打印出来观察完整过程：


In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置 DEEPSEEK_API_KEY"

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USER = "adk_ch03", "student"

async def run_verbose(agent, query, session_id="demo"):
    """运行一轮并打印完整事件流（工具调用过程可见）。"""
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=session_id)
    runner = Runner(agent=agent, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"🧑 {query}\n" + "─" * 55)
    async for ev in runner.run_async(user_id=USER, session_id=session_id, new_message=msg):
        if not (ev.content and ev.content.parts):
            continue
        for p in ev.content.parts:
            if p.function_call:
                print(f"🔧 [调用] {p.function_call.name}({dict(p.function_call.args)})")
            elif p.function_response:
                print(f"📦 [返回] {p.function_response.name} → {p.function_response.response}")
            elif p.text and ev.is_final_response():
                print(f"🤖 [回复] {p.text}")
    print("─" * 55)

# ---------- 定义两个工具 ----------
def get_weather(city: str) -> dict:
    """查询指定城市的实时天气。

    Args:
        city: 中国城市名，例如"北京"、"上海"。

    Returns:
        包含天气状况与温度的字典。
    """
    fake_db = {"北京": {"状况": "晴", "温度": "26°C"}, "上海": {"状况": "小雨", "温度": "22°C"}}
    return fake_db.get(city, {"状况": "未知（演示数据未收录）", "温度": "-"})

def calculate(expression: str) -> dict:
    """计算一个数学表达式并返回结果。支持加减乘除与括号。

    Args:
        expression: 数学表达式字符串，例如 "(3+5)*2"。
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {})  # 演示环境的安全沙箱写法
        return {"result": result}
    except Exception as e:
        return {"error": str(e)}

assistant = Agent(
    name="tool_demo",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是一个得力助手，需要数据时主动调用工具，不要自己编造。",
    description="带天气和计算工具的助手",
    tools=[get_weather, calculate],   # ← 函数直接放进列表
)

await run_verbose(assistant, "上海现在天气如何？如果穿短袖的建议温度是25度以上，(26-22)*3 是多少？")


🧑 上海现在天气如何？如果穿短袖的建议温度是25度以上，(26-22)*3 是多少？
───────────────────────────────────────────────────────


D:\Python\Lib\site-packages\google\adk\models\llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


22:55:55 - LiteLLM:WARNING: get_model_cost_map.py:289 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: _ssl.c:983: The handshake operation timed out. Falling back to local backup.


🔧 [调用] get_weather({'city': '上海'})
🔧 [调用] calculate({'expression': '(26-22)*3'})
📦 [返回] get_weather → {'状况': '小雨', '温度': '22°C'}
📦 [返回] calculate → {'result': 12}


🤖 [回复] 我来为您汇总一下：

### 🌧️ 上海当前天气
- **天气状况**：小雨
- **当前温度**：22°C

关于穿短袖的建议：您提到**穿短袖的建议温度是25°C以上**，而上海当前温度是 **22°C**，还没有达到25°C的建议标准。建议出门时携带外套或长袖，同时也别忘了带伞（因为在下小雨）。

### 🧮 数学计算
**(26 - 22) × 3 = 12**

---

总结：上海现在22°C，下着小雨，温度未达到穿短袖的25°C标准，建议添件外套并带伞。计算结果为 **12**。请问还有什么需要帮忙的吗？
───────────────────────────────────────────────────────


观察上面的输出，三个事件清晰可辨：`🔧 调用` → `📦 返回` → `🤖 回复`。LLM 还展现了**组合使用**两个工具的能力。

> ⚠️ **教学提醒**：docstring 不是写给人看的摆设——它是 LLM 的"使用说明书"。描述含糊是工具被误用/弃用的头号原因。

---

## 3. ToolContext：工具与状态的双向通道

普通函数是"无状态"的，但很多工具需要读写会话上下文（比如"记录用户偏好"、"统计调用次数"）。只要给函数加一个 `tool_context: ToolContext` 参数，ADK 就会**自动注入**上下文对象：

| 能力 | 写法 |
|---|---|
| 读状态 | `tool_context.state.get("key")` |
| 写状态 | `tool_context.state["key"] = value`（自动产生 state_delta 事件） |
| 会话信息 | `tool_context.session_id` / `tool_context.user_id` |
| 记忆检索 | `tool_context.search_memory(...)`（第 5 章） |


In [2]:
from google.adk.tools import ToolContext

def save_favorite(topic: str, tool_context: ToolContext) -> dict:
    """把用户喜欢的主题记录到会话状态中。

    Args:
        topic: 用户表示喜欢的主题，例如"科幻电影"。
    """
    favorites = tool_context.state.get("favorites", [])
    favorites.append(topic)
    tool_context.state["favorites"] = favorites   # 写入状态（会被持久化到 Session）
    return {"status": "已记录", "当前收藏": favorites}

def list_favorites(tool_context: ToolContext) -> dict:
    """查看当前会话中已记录的所有喜欢的主题。"""
    return {"favorites": tool_context.state.get("favorites", [])}

memory_assistant = Agent(
    name="memory_demo",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是贴心的兴趣管家：用户表达喜欢什么就调用 save_favorite 记录；用户问收藏就调用 list_favorites。",
    description="记录用户兴趣的管家",
    tools=[save_favorite, list_favorites],
)

await run_verbose(memory_assistant, "我特别喜欢科幻电影，对了还有爵士乐！帮我记下来，然后告诉我现在收藏了啥？", session_id="toolctx")


🧑 我特别喜欢科幻电影，对了还有爵士乐！帮我记下来，然后告诉我现在收藏了啥？
───────────────────────────────────────────────────────


🔧 [调用] save_favorite({'topic': '科幻电影'})
🔧 [调用] save_favorite({'topic': '爵士乐'})
📦 [返回] save_favorite → {'status': '已记录', '当前收藏': ['科幻电影', '爵士乐']}
📦 [返回] save_favorite → {'status': '已记录', '当前收藏': ['科幻电影', '爵士乐']}


🔧 [调用] list_favorites({})
📦 [返回] list_favorites → {'favorites': ['科幻电影', '爵士乐']}


🤖 [回复] 已经帮你记录好啦！你目前收藏的主题有：

1. **科幻电影** 🚀
2. **爵士乐** 🎷

这两个兴趣我都帮你记下来了，以后想聊科幻电影或者爵士乐的时候，随时都可以找我哦！还有其他喜欢的主题想记录吗？😊
───────────────────────────────────────────────────────


> 🔍 注意看：两次工具调用**共享了同一份 Session 状态**。这就是 `ToolContext` 的价值——让工具成为状态机的参与者，而非孤立函数。

---

## 4. AgentTool：把 Agent 变成工具

第 4 章会讲"层级委派"（父 Agent 把任务**转移**给子 Agent，控制权随之交出去）。但有时你想要的是另一种关系：**把专业 Agent 当作一个"可调用的函数"**——调用完还回到我这里继续干活。这正是 `AgentTool`：

```mermaid
flowchart LR
    subgraph MT["主 Agent（翻译项目经理）"]
        direction TB
        THINK["LLM 决策"]
    end
    subgraph AT["AgentTool 封装"]
        TRANS["翻译 Agent<br/>（英↔中专家）"]
    end
    THINK -->|"作为工具调用"| TRANS
    TRANS -->|"返回译文"| THINK
    style TRANS fill:#fce8e6,stroke:#ea4335
```


In [3]:
from google.adk.tools import AgentTool

translator = Agent(
    name="translator",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是专业翻译：输入中文则译成英文，输入英文则译成中文。只输出译文，不加解释。",
    description="中英互译专家",
)

pm = Agent(
    name="pm",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是项目经理：先用翻译工具把需求译成英文，再基于英文版本写一句推广语。",
    description="会调用翻译工具的项目经理",
    tools=[AgentTool(agent=translator)],   # ← 整个 Agent 变成一个工具
)

await run_verbose(pm, "需求：我们的新产品是一款能自动整理会议纪要的智能音箱。", session_id="agenttool")


🧑 需求：我们的新产品是一款能自动整理会议纪要的智能音箱。
───────────────────────────────────────────────────────


🔧 [调用] translator({'request': '我们的新产品是一款能自动整理会议纪要的智能音箱。'})


📦 [返回] translator → {'result': 'Our new product is a smart speaker that can automatically organize meeting minutes.'}


🤖 [回复] 好的，我已经成功将需求翻译成英文。现在基于这个英文版本，我来为您写一句推广语。

---

**需求翻译结果：**
> Our new product is a smart speaker that can automatically organize meeting minutes.

**推广语：**
> "Turn your meetings into minutes in a snap — our smart speaker captures and organizes it all, so you can focus on what matters."
───────────────────────────────────────────────────────


> 💡 **AgentTool vs sub_agents（预告）**：
> - `AgentTool`：**调用-返回**关系，主 Agent 始终握有控制权，适合"借用专业能力"；
> - `sub_agents` 转移：**移交-接管**关系，适合"这活儿归你负责"。
> 第 4 章会系统对比。

---

## 5. 内置工具与 MCP 生态

### 5.1 内置工具

ADK 预置了若干开箱工具，但要注意**模型兼容性**：

| 内置工具 | 用途 | 在 DeepSeek 下可用？ |
|---|---|---|
| `google_search` | Google 搜索接地 | ❌ 仅 Gemini 模型支持 |
| `BuiltInCodeExecutor` | 运行模型生成的代码 | ❌ 仅 Gemini 系列支持 |
| `VertexAiSearchTool` | 企业搜索 | 需 Vertex AI 凭证 |

> 📌 **规律**："**内置**"工具深度绑定 Gemini 的原生能力；使用第三方模型时，**FunctionTool / MCP / 第三方工具** 是你的主力。

### 5.2 MCPToolset：接入 MCP 工具宇宙

[Model Context Protocol（MCP）](https://modelcontextprotocol.io) 是 Anthropic 提出的工具开放协议，已成为事实标准。ADK 的 `MCPToolset` 可以把任意 MCP Server 的工具挂进来：

```python
from google.adk.tools.mcp_tool import MCPToolset
from mcp import StdioServerParameters

fs_tools = MCPToolset(
    connection_params=StdioServerParameters(
        command="npx",
        args=["-y", "@modelcontextprotocol/server-filesystem", "/tmp"],
    )
)
agent = Agent(..., tools=[fs_tools])
```

（本教程不实际启动 MCP Server，了解接入方式即可。）

### 5.3 第三方工具桥接

ADK 还能直接复用其他框架的工具生态：

- **`LangchainTool`**：包装任意 LangChain 工具（如 `TavilySearchResults`）；
- **`CrewaiTool`**：包装 CrewAI 工具（如 `SerperDevTool`）。

这体现 ADK 的务实策略：**不必重复造轮子，把全行业的工具都变成自己的生态**。

---

## 6. 长时任务：LongRunningFunctionTool

有些工具一跑就是几分钟（视频渲染、审批流程）。`LongRunningFunctionTool` 让工具"先挂起、后回填"：Agent 发起调用后不必干等，结果就绪时框架再通过事件流续上。这是 ADK 支持**异步人机协同**的基础构件，第 6 章结合回调再谈。

---

## 7. 与 LangChain 对照 🔄

| ADK | LangChain / LangGraph | 差异点评 |
|---|---|---|
| 函数直接放入 `tools=[...]` | `@tool` 装饰器 或 `create_agent(tools=[函数])` | LangChain 1.x 也已支持直接传函数，体验趋同 |
| docstring 即工具描述 | docstring 或 `@tool(description=...)` | 一致 |
| `ToolContext` 注入 | `ToolRuntime` / `InjectedState` 注入 | 思想相同：工具访问图状态 |
| `AgentTool` | 把子图封装为工具 / supervisor 模式 | LangGraph 里"Agent 即节点"，更自由也更手动 |
| `MCPToolset` | `langchain-mcp-adapters` | 都拥抱 MCP 标准 |
| 内置工具绑 Gemini | 集成包绑各服务商 | 都要注意模型兼容性 |

---

## 📌 本章要点回顾

- 函数 + **docstring** + **类型注解** = 一个合格的 ADK 工具；
- 工具事件流：`function_call` → `function_response` → 最终回复；
- `ToolContext` 让工具读写 Session 状态；
- `AgentTool` = "调用-返回"式的 Agent 复用；
- 第三方模型下以 FunctionTool / MCP / 桥接工具为主力，内置工具多为 Gemini 专属。

> ➡️ 下一章：[04-多智能体与工作流](04-多智能体与工作流.ipynb) —— ADK 最精彩的部分。
